# Задача № 3. Военный или гражданский самолёт по фотографии

**Статус: данные подготовлены и проверены; выполнение моделей ожидает пополнения кредитов HF.**
Проверка 24.10.2026: токен принят, запрос вернул HTTP 402 «no remaining credits». Предсказаний не получено.
Численные результаты и вывод появятся после запуска с HF_TOKEN. Предсказания не выдуманы.

Сравниваем две мультимодальные LLM (VLM) через **Hugging Face Inference Providers**:
`Qwen/Qwen2.5-VL-72B-Instruct` и `zai-org/GLM-4.5V`.
На вход подаётся только JPEG и одинаковый промпт, без названия самолёта, эталона и URL источника.

## 1. Подготовка среды
В Jupyter откройте notebook из корня проекта. В Colab загрузите ZIP проекта в Files,
распакуйте его в `/content` и откройте этот notebook. GPU не нужен.
Добавьте **HF_TOKEN** в Colab Secrets с разрешением **Make calls to Inference Providers**.
Токен нельзя публиковать в коде или выводе. Локально он запрашивается скрытым вводом.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if not (ROOT / 'src/experiment.py').exists():
    found = list(ROOT.glob('**/aircraft-hf-comparison/src/experiment.py'))
    if len(found) != 1:
        raise RuntimeError('Распакуйте проект и перейдите в его папку.')
    ROOT = found[0].parents[1]
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print(ROOT)

In [ ]:
%pip -q install -r requirements.txt

In [ ]:
# Для просмотра отчёта токен не нужен. Получаем его только при RUN_INFERENCE=True.
RUN_INFERENCE = False  # После пополнения баланса измените на True.
if RUN_INFERENCE:
    import getpass
    if not os.environ.get('HF_TOKEN'):
        try:
            from google.colab import userdata
            os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        except Exception:
            os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN (ввод скрыт): ')


## 2. Определение класса и выборка
Классифицируется **основное назначение изображённой модификации**, а не владелец конкретного борта.
Военный класс включает учебные, транспортные, патрульные и самолёты-заправщики без оружия.
Гражданский — пассажирские, грузовые, частные, спортивные, сельскохозяйственные и пожарные самолёты.
Военный оператор сам по себе не меняет назначение базовой гражданской конструкции.

30 разных типов, по 15 на класс. 20 заранее отмечены как сложные: военные платформы на базе авиалайнеров,
гражданский L-100 на основе военного Hercules, спортивные самолёты и специальные грузовые конструкции.
Ещё 10 — контрольные примеры. Разбиение на обучение не нужно: используется zero-shot без обучения.
Промпт и сложность зафиксированы **до** предсказаний; ответы не используются для изменения эталона.

Эталон сначала составлен по назначению типа. Каждая фотография требует проверки:
совпадает ли модификация, виден ли самолёт, это фото, а не схема или коллаж, нет ли дублей.
При несоответствии выберите другое фото того же типа до запросов, обновив источник, лицензию и SHA-256.
Например, не путайте гражданский AT-802 с вооружённым AT-802U.

In [1]:
import pandas as pd
from experiment import prepare, checked_rows, run, report, PROMPT
selection = pd.read_csv('data/selection.csv', dtype={'id': str})
display(selection)
display(pd.crosstab(selection.label, selection.hard))
prepare()  # Зафиксированный manifest повторно не скачивается.

,id,page_title,label,hard,reason
0,01,General Dynamics F-16 Fighting Falcon,military,0,Типичный истребитель
1,02,Sukhoi Su-27,military,0,Типичный истребитель
2,03,Lockheed Martin F-35 Lightning II,military,0,Типичный истребитель
3,04,Northrop B-2 Spirit,military,0,Бомбардировщик необычной формы
4,05,Fairchild Republic A-10 Thunderbolt II,military,0,Штурмовик
5,06,Boeing C-17 Globemaster III,military,1,Военный транспорт без вооружения
6,07,Lockheed C-5 Galaxy,military,1,Крупный транспорт похож на гражданский грузовой
7,08,Airbus A400M Atlas,military,1,Турбовинтовой транспорт
8,09,Boeing KC-135 Stratotanker,military,1,Заправщик похож на авиалайнер
9,10,Airbus A330 MRTT,military,1,Военная версия гражданской платформы


hard,0,1
label,,
civilian,5,10
military,5,10


Manifest already frozen


### Галерея, источники и ручная проверка
Основные свободные иллюстрации статей выбираются Wikipedia API; Commons API сохраняет авторство,
описание и лицензию каждого файла. JPEG уменьшены до 960 px, метаданные модели не передаются.
Перейдите по ссылкам и проверьте назначение модификации. Автоматическая загрузка не заменяет разметку.

In [1]:
from IPython.display import display, HTML, Image
from html import escape
manifest = pd.read_csv('data/manifest.csv', dtype=str).fillna('')
for r in manifest.to_dict('records'):
    display(HTML(f"<h4>{escape(r['id'])}: {escape(r['page_title'])} — {escape(r['label'])}</h4>"
        f"<p>{escape(r['reason'])}</p><a href='{escape(r['article_url'])}'>Назначение</a> · "
        f"<a href='{escape(r['file_url'])}'>Источник и лицензия</a><p>{escape(r['license'])}</p>"))
    display(Image(filename=r['image_path'], width=480))


01: General Dynamics F-16 Fighting Falcon — military Типичный истребитель Назначение · Источник и лицензия Public domain

02: Sukhoi Su-27 — military Типичный истребитель Назначение · Источник и лицензия GFDL 1.2

03: Lockheed Martin F-35 Lightning II — military Типичный истребитель Назначение · Источник и лицензия Public domain

04: Northrop B-2 Spirit — military Бомбардировщик необычной формы Назначение · Источник и лицензия OGL v1.0

05: Fairchild Republic A-10 Thunderbolt II — military Штурмовик Назначение · Источник и лицензия Public domain

...

26: Bede BD-5 — civilian Спортивный, необычный силуэт Назначение · Источник и лицензия CC0

27: Pilatus PC-12 — civilian Гражданский тип со специальными операторами Назначение · Источник и лицензия GFDL 1.2

28: Piaggio P.180 Avanti — civilian Необычная компоновка Назначение · Источник и лицензия CC BY-SA 3.0

29: Airbus Beluga — civilian Специальный грузовой Назначение · Источник и лицензия CC BY-SA 4.0

30: Boeing Dreamlifter — civilian Крупногабаритный грузовой Назначение · Источник и лицензия CC BY-SA 2.0

In [1]:
# Только после ручной проверки: впишите подтверждённые ID, например ['01', '02'].
# Уже подтверждённые строки сохраняются.
CONFIRMED_IDS = []
if CONFIRMED_IDS:
    manifest.loc[manifest.id.isin(CONFIRMED_IDS), 'reviewed'] = '1'
    manifest.to_csv('data/manifest.csv', index=False)
print('Подтверждено:', (manifest.reviewed == '1').sum(), '/ 30')

Подтверждено: 30 / 30


## 3. Протокол и бюджет
Одинаковые фотографии и промпт, temperature=0, max_tokens=160. Очерёдность моделей чередуется.
Проверяется live-провайдер на Hub для мультимодальных моделей (task conversational или image-text-to-text).
Провайдеры закреплены: OVHcloud для Qwen и Novita для GLM; фактический выбор сохраняется в журнале.
Поддерживается и auto: первый доступный по алфавиту. Изменение провайдера создаёт другой эксперимент.
Если модель недоступна, замените её **до** эксперимента и отразите замену в отчёте.
Это сравнение моделей с обслуживающими их провайдерами, а не контролируемый тест одинакового оборудования.

План — 60 ответов (30 × 2). Повторный запуск пропускает сохранённые ответы. Автоповторов нет.
Ошибка API останавливает запуск; некорректный JSON сохраняется как invalid и считается ошибкой.
Лимит 60 ограничивает попытки текущего запуска, а не стоимость. Неуспешные попытки тоже могут расходовать
кредит. Бесплатный лимит HF денежный и может не покрыть все запросы; сначала проверьте баланс.

In [ ]:
import json
print(PROMPT)
display(json.loads(Path('config.json').read_text()))
checked_rows()
# Для пробного запуска используйте run(limit=2).
if RUN_INFERENCE:
    run()
else:
    print('Запросы выключены. После пополнения HF установите RUN_INFERENCE=True.')


## 4. Метрики и анализ ошибок
Основная метрика — **macro-F1** двух классов. Дополнительно: accuracy, macro-precision,
macro-recall, accuracy на 20 сложных случаях, coverage корректных ответов и среднее/медианное время.
Invalid считается промахом и false negative своего истинного класса; он не удаляется из знаменателя.
Матрица ошибок включает отдельную колонку invalid. Транспортные ошибки не являются предсказаниями:
для сравнения нужно завершить все 60 ответов. Промежуточные результаты победителя не определяют.

Точный парный тест Мак-Немара сравнивает несовпадающие успехи моделей на тех же фото.
На 30 примерах один ответ меняет accuracy на 3,33 п.п.; небольшая разница не доказывает устойчивое превосходство.
Задержка включает сеть и нагрузку провайдера. Token usage сохраняется при наличии, но не равен стоимости.

In [1]:
from IPython.display import Image
results_ready = Path('results/predictions.jsonl').exists()
if results_ready:
    metrics, errors, summary = report()  # Требует ровно 60 ответов.
    display(metrics.round(4))
    display(Image(filename='results/confusion_matrices.png'))
    display(errors[['id','page_title','model','truth','prediction','raw','reason']])
    print(summary)
else:
    print('Метрики пока отсутствуют: HF вернул 402, доступные кредиты закончились.')


Метрики пока отсутствуют: HF вернул 402, доступные кредиты закончились.


## 5. Вывод по реальным данным
Следующая ячейка формирует вывод по сохранённым ответам. После запуска дополните его разбором
2–3 ошибок из таблицы: какое визуальное сходство привело к неверному классу, каково назначение модификации.
До завершения запросов численный вывод не сформирован.

In [ ]:
if results_ready:
    from IPython.display import Markdown
    best, other = metrics.iloc[0], metrics.iloc[1]
    text = ('Одинаковый macro-F1.' if abs(best.f1_macro-other.f1_macro)<1e-12 else
            f"На выбранных фото выше macro-F1 у **{best['model']}**: {best.f1_macro:.3f} против {other.f1_macro:.3f}.")
    display(Markdown(text + f" Тест Мак-Немара: p={summary['mcnemar_exact_p']:.4f}. "
        'Эта небольшая целевая выборка не позволяет обобщить результат на все фотографии самолётов.'))
else:
    print('Сравнительный вывод будет сформирован после 60 реальных ответов.')

## 6. Ограничения и воспроизводимость
- Выборка намеренно сложная и сбалансированная; частоты не соответствуют реальному потоку.
- Фото Wikipedia могли присутствовать в обучающих данных моделей; утечка не исключена.
- Назначение конкретной модификации иногда невозможно однозначно восстановить по одному кадру.
- Версии серверов, провайдеры и недетерминированность могут влиять на ответы при temperature=0.
- Журнал хранит модели/провайдеры, UTC-время, исходный ответ, usage, SHA-256 JPEG и подпись эксперимента.
- Источники, авторы и лицензии — в `data/manifest.csv`. При публикации сохраняйте атрибуцию;
  код и фотографии имеют разные лицензии. Фотографии уменьшены и перекодированы.

## Источники
1. [HF image-text-to-text](https://huggingface.co/docs/inference-providers/en/tasks/image-text-to-text)
2. [HF pricing](https://huggingface.co/docs/inference-providers/en/pricing)
3. [Qwen2.5-VL-72B](https://huggingface.co/Qwen/Qwen2.5-VL-72B-Instruct)
4. [GLM-4.5V](https://huggingface.co/zai-org/GLM-4.5V)
5. Страницы Wikipedia и Wikimedia Commons для каждого фото — manifest.csv.

Документация проверена 24.10.2026. Актуальная доступность моделей проверяется перед запросами.

## Приложение. Полный код эксперимента

Рабочая копия находится в `src/experiment.py`.

```python
import argparse,base64,csv,hashlib,io,json,os,time
from datetime import datetime,timezone
from pathlib import Path
ROOT=Path(__file__).resolve().parents[1]
PROMPT='''Classify the largest, primary aircraft in the photo by the primary intended role of its variant.
military: designed primarily for armed forces, including unarmed trainers,
transports, tankers and patrol aircraft. civilian: designed primarily for passenger,
cargo, private, agricultural, firefighting or sport aviation. A military operator
alone does not change a civilian design's class. Choose the most likely class
from visual evidence. Return ONLY JSON with keys label (military or civilian)
and reason (short visual explanation). Do not use outside tools.'''
def read_csv(p):
 with Path(p).open(encoding='utf-8',newline='') as f:return list(csv.DictReader(f))
def write_csv(p,rows):
 p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
 with p.open('w',encoding='utf-8',newline='') as f:
  w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def prepare():
 import requests
 from PIL import Image,ImageOps
 manifest=ROOT/'data/manifest.csv'
 if manifest.exists():print('Manifest already frozen');return
 session=requests.Session();session.headers['User-Agent']='AircraftCourseProject/1.0 (educational image classification)'
 def get(url,**kw):
  for attempt in range(5):
   r=session.get(url,timeout=90,**kw)
   if r.status_code not in (429,503):r.raise_for_status();return r
   time.sleep(min(30,5*(attempt+1)))
  r.raise_for_status()
 def api(url,**kw):return get(url,params={'action':'query','format':'json',**kw}).json()['query']['pages']
 selection=read_csv(ROOT/'data/selection.csv');cache=ROOT/'data/source_cache.json'
 if cache.exists():sources=json.loads(cache.read_text(encoding='utf-8'))
 else:
  pages=api('https://en.wikipedia.org/w/api.php',titles='|'.join(r['page_title'] for r in selection),redirects=1,prop='pageimages',piprop='name|thumbnail',pithumbsize=960,pilicense='free')
  sources={p['title']:p for p in pages.values()}
  cache.write_text(json.dumps(sources,ensure_ascii=False),encoding='utf-8')
 cc=ROOT/'data/commons_cache.json'
 if cc.exists():commons=json.loads(cc.read_text(encoding='utf-8'))
 else:
  commons=api('https://commons.wikimedia.org/w/api.php',titles='|'.join('File:'+p['pageimage'] for p in sources.values()),prop='imageinfo',iiprop='url|extmetadata')
  cc.write_text(json.dumps(commons,ensure_ascii=False),encoding='utf-8')
 infos={p['title'].removeprefix('File:'):p['imageinfo'][0] for p in commons.values() if 'imageinfo' in p}
 rows=[];(ROOT/'data/images').mkdir(exist_ok=True)
 for row in selection:
  p=sources.get(row['page_title'])
  if p is None:
   match=[v for k,v in sources.items() if k.lower()==row['page_title'].lower()]
   if not match:raise RuntimeError('Redirect: adjust selection title '+row['page_title'])
   p=match[0]
  if 'thumbnail' not in p:raise RuntimeError('No free lead photo: '+row['page_title'])
  info=infos[p['pageimage'].replace('_',' ')];meta=info.get('extmetadata',{});field=lambda k:meta.get(k,{}).get('value','')
  if not field('LicenseShortName'):raise RuntimeError('Missing license')
  path=ROOT/'data/images'/(row['id']+'.jpg')
  if not path.exists():
   r=get(p['thumbnail']['source']);im=ImageOps.exif_transpose(Image.open(io.BytesIO(r.content))).convert('RGB');im.thumbnail((960,960));im.save(path,quality=90);time.sleep(1)
  rows.append({**row,'image_path':path.relative_to(ROOT).as_posix(),'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'article_url':'https://en.wikipedia.org/wiki/'+p['title'].replace(' ','_'),'file_url':info['descriptionurl'],'download_url':p['thumbnail']['source'],'author':field('Artist'),'license':field('LicenseShortName'),'license_url':field('LicenseUrl'),'description':field('ImageDescription'),'reviewed':'0'})
  print(row['id'],row['page_title'],flush=True)
 write_csv(manifest,rows)
def checked_rows():
 rows=read_csv(ROOT/'data/manifest.csv')
 assert len(rows)==30 and len({r['id'] for r in rows})==30
 assert sum(r['label']=='military' for r in rows)==15 and sum(r['label']=='civilian' for r in rows)==15
 assert all(r['reviewed']=='1' for r in rows),'Review every photo and set reviewed=1 first'
 hashes=[]
 for r in rows:
  h=hashlib.sha256((ROOT/r['image_path']).read_bytes()).hexdigest();assert h==r['sha256'];hashes.append(h)
 assert len(set(hashes))==30,'Duplicate photos'
 return rows
def parse_label(raw):
 d=json.loads(raw)
 if not isinstance(d,dict) or d.get('label') not in ('military','civilian'):raise ValueError('Invalid schema')
 return d['label']
def run(limit=None):
 from huggingface_hub import HfApi,InferenceClient
 rows=checked_rows();token=os.environ.get('HF_TOKEN')
 if not token:
  from huggingface_hub import get_token
  token=get_token()
 if not token:raise RuntimeError('Set HF_TOKEN securely')
 config=json.loads((ROOT/'config.json').read_text());assert len(config['models'])==2
 assert len({m['id'] for m in config['models']})==2
 for m in config['models']:
  info=HfApi(token=token).model_info(m['id'],expand=['inferenceProviderMapping'])
  mapping=info.inference_provider_mapping or []
  pairs=mapping.items() if isinstance(mapping,dict) else ((v.provider,v) for v in mapping)
  available={k:v for k,v in pairs if v.status=='live' and v.task in ('image-text-to-text','conversational')}
  if not available:raise RuntimeError('No live vision provider: '+m['id'])
  if m['provider']=='auto':m['provider']=sorted(available)[0]
  if m['provider'] not in available:raise RuntimeError('Provider unavailable')
 out=ROOT/'results';out.mkdir(exist_ok=True);log=out/'predictions.jsonl'
 signature=hashlib.sha256(json.dumps({'config':config,'prompt':PROMPT,'data':[(r['id'],r['sha256'],r['label'],r['hard']) for r in rows]},sort_keys=True).encode()).hexdigest()
 old=[json.loads(s) for s in log.read_text(encoding='utf-8').splitlines()] if log.exists() else []
 if any(r['experiment']!=signature for r in old):raise RuntimeError('Experiment changed: archive old results first')
 done={(r['model'],r['id']) for r in old};count=0
 budget=config['max_new_requests'] if limit is None else min(limit,config['max_new_requests'])
 clients={m['id']:InferenceClient(provider=m['provider'],api_key=token,timeout=120) for m in config['models']}
 for i,row in enumerate(rows):
  for m in config['models'][::1 if i%2==0 else -1]:
   if (m['id'],row['id']) in done:continue
   if count>=budget:print('Request limit reached; resume later');return
   encoded=base64.b64encode((ROOT/row['image_path']).read_bytes()).decode()
   messages=[{'role':'user','content':[{'type':'text','text':PROMPT},{'type':'image_url','image_url':{'url':'data:image/jpeg;base64,'+encoded}}]}]
   rec={'experiment':signature,'id':row['id'],'model':m['id'],'provider':m['provider'],'truth':row['label'],'hard':int(row['hard']),'sha256':row['sha256'],'timestamp':datetime.now(timezone.utc).isoformat()}
   start=time.perf_counter();count+=1
   try:response=clients[m['id']].chat.completions.create(model=m['id'],messages=messages,temperature=config['temperature'],max_tokens=config['max_tokens'])
   except Exception as e:raise RuntimeError('HF request stopped ('+type(e).__name__+'); prior answers saved; no automatic retries') from None
   rec['latency_s']=time.perf_counter()-start;rec['raw']=response.choices[0].message.content or ''
   rec['usage']=dict(response.usage) if response.usage is not None else None
   try:rec['prediction']=parse_label(rec['raw']);rec['status']='ok'
   except (ValueError,TypeError):rec['prediction']='invalid';rec['status']='invalid'
   with log.open('a',encoding='utf-8') as f:f.write(json.dumps(rec,ensure_ascii=False,default=str)+'\n')
   print(row['id'],m['id'],rec['prediction']);time.sleep(.5)
def report():
 import pandas as pd
 import matplotlib.pyplot as plt
 from sklearn.metrics import accuracy_score,precision_recall_fscore_support,confusion_matrix
 rows=checked_rows();path=ROOT/'results/predictions.jsonl'
 if not path.exists():raise RuntimeError('No real predictions yet')
 df=pd.read_json(path,lines=True,dtype={'id':str})
 expected={m['id'] for m in json.loads((ROOT/'config.json').read_text())['models']}
 assert len(df)==60 and set(df.model)==expected,'Complete all 60 responses before comparing'
 assert not df.duplicated(['model','id']).any() and df.experiment.nunique()==1
 assert all(set(g.id)=={r['id'] for r in rows} for _,g in df.groupby('model'))
 metrics=[];fig,axes=plt.subplots(1,2,figsize=(12,4))
 for ax,(model,g) in zip(axes,df.groupby('model')):
  p,r,f,_=precision_recall_fscore_support(g.truth,g.prediction,labels=['military','civilian'],average='macro',zero_division=0)
  metrics.append({'model':model,'n':len(g),'accuracy':accuracy_score(g.truth,g.prediction),'precision_macro':p,'recall_macro':r,'f1_macro':f,'coverage':g.prediction.isin(['military','civilian']).mean(),'hard_accuracy':(g[g.hard==1].truth==g[g.hard==1].prediction).mean(),'mean_latency_s':g.latency_s.mean(),'median_latency_s':g.latency_s.median()})
  cm=confusion_matrix(g.truth,g.prediction,labels=['military','civilian','invalid'])[:2,:]
  ax.imshow(cm,cmap='Blues');ax.set_xticks(range(3),['military','civilian','invalid']);ax.set_yticks(range(2),['military','civilian']);ax.set_xlabel('Prediction');ax.set_ylabel('Truth');ax.set_title(model.split('/')[-1])
  for y in range(2):
   for x in range(3):ax.text(x,y,str(cm[y,x]),ha='center',va='center')
 out=ROOT/'results';fig.tight_layout();fig.savefig(out/'confusion_matrices.png',dpi=160);plt.close(fig)
 table=pd.DataFrame(metrics).sort_values('f1_macro',ascending=False);table.to_csv(out/'metrics.csv',index=False)
 errors=df[df.truth!=df.prediction].merge(pd.DataFrame(rows)[['id','page_title','reason','file_url']],on='id');errors.to_csv(out/'errors.csv',index=False)
 a,b=sorted(expected);pv=df.pivot(index='id',columns='model',values='prediction');truth=df[df.model==a].set_index('id').truth
 ca=pv[a].eq(truth);cb=pv[b].eq(truth);n10=int((ca & ~cb).sum());n01=int((~ca & cb).sum());n=n10+n01
 from math import comb
 pvalue=min(1.,2*sum(comb(n,k) for k in range(min(n10,n01)+1))/2**n) if n else 1.
 summary={'status':'complete','n_images':30,'n_responses':60,'a':a,'b':b,'a_only_correct':n10,'b_only_correct':n01,'mcnemar_exact_p':pvalue}
 (out/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
 print(table.to_string(index=False));return table,errors,summary
if __name__=='__main__':
 p=argparse.ArgumentParser();p.add_argument('command',choices=['prepare','run','report']);p.add_argument('--limit',type=int);a=p.parse_args()
 if a.command=='prepare':prepare()
 elif a.command=='run':run(a.limit)
 else:report()

```